本节的学习目的只在于学习基础的ROS2语法，对于更细节的部分留在以后项目过程中逐步学习，一切以先上手为主

一键安装ROS2：wget http://fishros.com/install -O fishros && . fishros

In [ ]:
#ROS2核心在于ROS节点，每一个节点只单独负责一个单独的模块功能
#节点运行命令  ros2 run <packag_name> <executable_name>
#查看节点列表  ros2 node list
#查看节点信息  ros2 node info <node_name>
#重映射节点名  ros2 run <packag_name> <executable_name> --ros-args --remap __node:=<new_name>

#工作空间，功能包，节点三者是前一个包含后一个的关系
#创建功能包  ros2 pkg create <pkg_name> --build-type {cmake/ament_python/ament_cmake}  --dependencies <依赖名称>
#查看功能包可执行文件 ros2 pkg executables <功能包名称>
#查看功能包所有包  ros2 pkg list <功能包名称>
#查看功能包前缀    ros2 pkg prefix <功能包名称>
#查看功能包信息    ros2 pkg xml <功能包名称>

#colcon是工作空间的构建工具，会自己生成build之类的文件夹

#节点demo文件 c++
#include "rclcpp/rclcpp.hpp"

int main(int argc, char **argv)
{
    //初始化
    rclcpp::init(argc,argv);
    //生成一个节点
    auto node = std::make_shared<rclcpp::Node>("node_01");
    //打印一句自我介绍
    RCLCPP_INFO(node->get_logger(), "node节点启动");
    //运行节点并检测ctrl c
    rclcpp::spin(node);
    //停止运行
    rclcpp::shutdown();
    return 0;
}

#完成上述文件后还需要更改CmakeLists.txt
#add_executable(包名 src/名称.cpp)
#ament_target_dependencies(名称 rclcpp)
#install(TARGETS
#  包名
#  DESTINATION lib/${PROJECT_NAME}
#)

#在工作空间下执行 colcon build构建工作空间，然后就可以运行
#在实际应用中主要还是把node包装成类，用类去实现节点功能

#ROS2节点的通信共有四种
    #话题，服务，动作，参数

#话题 主要就是发布话题和订阅话题
#发布者代码
#include "rclcpp/rclcpp.hpp"
#include "std_msgs/msg/string.hpp"

class TopicPublisher01 : public rclcpp::Node
{
public:
    TopicPublisher01(std::string name) : Node(name)
    {
        RCLCPP_INFO(this->get_logger(), "大家好，我是%s.", name.c_str());
        //发布者
        command_publisher_ = this.create_publisher<std_msgs::msg::String>("command", 10);
        //定时器
        timer_ = this->create_wall_timer(std::chrono::milliseconds(500), std::bind(&TopicPublisher01::timer_callback, this));
    }
private:
    void timer_callback()
    {
        // 创建消息
        std_msgs::msg::String message;
        message.data = "forward";
        // 日志打印
        RCLCPP_INFO(this->get_logger(), "Publishing: '%s'", message.data.c_str());
        // 发布消息
        command_publisher_->publish(message);
    }

    // 声名定时器指针
    rclcpp::TimerBase::SharedPtr timer_;
    // 声明话题发布者指针
    rclcpp::Publisher<std_msgs::msg::String>::SharedPtr command_publisher_;
}

#订阅者代码
#include "rclcpp/rclcpp.hpp"
#include "std_msgs/msg/string.hpp"

class TopicSubscribe01 : public rclcpp::Node
{
public:
    TopicSubscribe01(std::string name) : Node(name)
    {
        RCLCPP_INFO(this->get_logger(), "大家好，我是%s.", name.c_str());
          // 创建一个订阅者订阅话题
        command_subscribe_ = this->create_subscription<std_msgs::msg::String>("command", 10, std::bind(&TopicSubscribe01::command_callback, this, std::placeholders::_1));
    }

private:
     // 声明一个订阅者
    rclcpp::Subscription<std_msgs::msg::String>::SharedPtr command_subscribe_;
     // 收到话题数据的回调函数
    void command_callback(const std_msgs::msg::String::SharedPtr msg)
    {
        double speed = 0.0f;
        if(msg->data == "forward")
        {
            speed = 0.2f;
        }
        RCLCPP_INFO(this->get_logger(), "收到[%s]指令，发送速度 %f", msg->data.c_str(),speed);
    }
};

#服务 分为请求和响应

#手动调用服务：ros2 service call 服务 client "{x:参数}"

#服务端代码
#include "example_interfaces/srv/add_two_ints.hpp"
#include "rclcpp/rclcpp.hpp"

class ServiceServer01 : public rclcpp::Node {
public:
  ServiceServer01(std::string name) : Node(name) {
    RCLCPP_INFO(this->get_logger(), "节点已启动：%s.", name.c_str());
    // 创建服务
    add_ints_server_ =
      this->create_service<example_interfaces::srv::AddTwoInts>(
        "add_two_ints_srv",
        std::bind(&ServiceServer01::handle_add_two_ints, this,
                  std::placeholders::_1, std::placeholders::_2));
  }

private:
  // 声明一个服务
  rclcpp::Service<example_interfaces::srv::AddTwoInts>::SharedPtr
    add_ints_server_;

  // 收到请求的处理函数
  void handle_add_two_ints(
    const std::shared_ptr<example_interfaces::srv::AddTwoInts::Request> request,
    std::shared_ptr<example_interfaces::srv::AddTwoInts::Response> response) {
    RCLCPP_INFO(this->get_logger(), "收到a: %ld b: %ld", request->a,
                request->b);
    response->sum = request->a + request->b;
  };
};

#客户端代码
#include "example_interfaces/srv/add_two_ints.hpp"

class ServiceClient01 : public rclcpp::Node {
public:
  // 构造函数,有一个参数为节点名称
  ServiceClient01(std::string name) : Node(name) {
    RCLCPP_INFO(this->get_logger(), "节点已启动：%s.", name.c_str());
    // 创建客户端
    client_ = this->create_client<example_interfaces::srv::AddTwoInts>("add_two_ints_srv");
  }

  void send_request(int a, int b) {
    RCLCPP_INFO(this->get_logger(), "计算%d+%d", a, b);

    // 1.等待服务端上线
    while (!client_->wait_for_service(std::chrono::seconds(1))) {
      //等待时检测rclcpp的状态
      if (!rclcpp::ok()) {
        RCLCPP_ERROR(this->get_logger(), "等待服务的过程中被打断...");
        return;
      }
      RCLCPP_INFO(this->get_logger(), "等待服务端上线中");
    }

    // 2.构造请求的
    auto request =
      std::make_shared<example_interfaces::srv::AddTwoInts_Request>();
    request->a = a;
    request->b = b;

    // 3.发送异步请求，然后等待返回，返回时调用回调函数
    client_->async_send_request(
      request, std::bind(&ServiceClient01::result_callback_, this,
                         std::placeholders::_1));
  };

private:
  // 声明客户端
  rclcpp::Client<example_interfaces::srv::AddTwoInts>::SharedPtr client_;

  void result_callback_(
    rclcpp::Client<example_interfaces::srv::AddTwoInts>::SharedFuture
      result_future) {
    auto response = result_future.get();
    RCLCPP_INFO(this->get_logger(), "计算结果：%ld", response->sum);
  }
};

#接口，数据传输的规范，话题接口：xxx.msg  服务接口：xxx.srv  动作接口 xxx.action
#上面这些接口最后都会通过IDL模块转换成头文件
#创建的时候同时要在Cmakelist里面声明这些接口，让idl转成头文件

#参数通信，ros2中的参数由key - value对组成，简单的理解就是运行节点的时候后面跟上一堆的参数就是参数通信
#也可以通过 ros2 param set <node_name> <param_key> <param_value>的方式改变

#使用参数通信
#1.用declare_parameter显示声明
#2.用get_parameter将参数获取到本地变量
#3.然后正常使用变量
#include <chrono>
#include "rclcpp/rclcpp.hpp"
/*
    # declare_parameter            声明和初始化一个参数
    # describe_parameter(name)  通过参数名字获取参数的描述
    # get_parameter                通过参数名字获取一个参数
    # set_parameter                设置参数的值
*/
class ParametersBasicNode : public rclcpp::Node {
 public:
  // 构造函数,有一个参数为节点名称
  explicit ParametersBasicNode(std::string name) : Node(name) {
    RCLCPP_INFO(this->get_logger(), "节点已启动：%s.", name.c_str());
    this->declare_parameter("rcl_log_level", 0);     /*声明参数*/
    this->get_parameter("rcl_log_level", log_level); /*获取参数*/
    /*设置日志级别*/
    this->get_logger().set_level((rclcpp::Logger::Level)log_level);
    using namespace std::literals::chrono_literals;
    timer_ = this->create_wall_timer(
        500ms, std::bind(&ParametersBasicNode::timer_callback, this));
  }

 private:
  int log_level;
  rclcpp::TimerBase::SharedPtr timer_;

  void timer_callback() {
    this->get_parameter("rcl_log_level", log_level); /*获取参数*/
    /*设置日志级别*/
    this->get_logger().set_level((rclcpp::Logger::Level)log_level);
    std::cout<<"======================================================"<<std::endl;
    RCLCPP_DEBUG(this->get_logger(), "我是DEBUG级别的日志，我被打印出来了!");
    RCLCPP_INFO(this->get_logger(), "我是INFO级别的日志，我被打印出来了!");
    RCLCPP_WARN(this->get_logger(), "我是WARN级别的日志，我被打印出来了!");
    RCLCPP_ERROR(this->get_logger(), "我是ERROR级别的日志，我被打印出来了!");
    RCLCPP_FATAL(this->get_logger(), "我是FATAL级别的日志，我被打印出来了!");
  }
};

int main(int argc, char** argv) {
  rclcpp::init(argc, argv);
  /*创建对应节点的共享指针对象*/
  auto node = std::make_shared<ParametersBasicNode>("parameters_basic");
  /* 运行节点，并检测退出信号*/
  rclcpp::spin(node);
  rclcpp::shutdown();
  return 0;
}

#Action包含三个组成：
#目标：即Action客户端告诉服务端要做什么，服务端针对该目标要有响应。解决了不能确认服务端接收并处理目标问题
#反馈：即Action服务端告诉客户端此时做的进度如何（类似与工作汇报）。解决执行过程中没有反馈问题
#结果：即Action服务端最终告诉客户端其执行结果，结果最后返回，用于表示任务最终执行情况。
#Action本质是由话题和服务组成的
#Action的示例代码太长，基本逻辑也就是create_server()创建一个服务，create_client()创建一个客户，然后绑定回调函数，没有新的东西


以上基本就是ROS2的所有的基础语法知识

下面是ROS2中常见的工具学习

In [ ]:
#Launch启动工具
#launch文件是用来启动一堆节点的文件，以.launch结尾，是一种xml文件
#launch文件有三种编写格式py,xml,yaml,其他两种没什么好说的，不同的标签就是不同的功能
#用py编写launch需要后缀改为.launch.py
#我们需要导入两个库，一个叫做LaunchDescription，用于对launch文件内容进行描述，一个是Node，用于声明节点所在的位置。
# 导入库
from launch import LaunchDescription
from launch_ros.actions import Node

#注意这里要定一个名字叫做generate_launch_description的函数，ROS2会对该函数名字做识别。
def generate_launch_description():
    """launch内容描述函数，由ros2 launch 扫描调用"""
    action_robot_01 = Node(
        package="example_action_rclcpp",
        executable="action_robot_01"
    )
    action_control_01 = Node(
        package="example_action_rclcpp",
        executable="action_control_01"
    )
    # 创建LaunchDescription对象launch_description,用于描述launch文件
    launch_description = LaunchDescription(
        [action_robot_01, action_control_01])
    # 返回让ROS2根据launch描述执行节点
    return launch_description

#RVIZ
#RVIZ2是ROS2中的一个非常重要且常用的数据可视化工具。
#一个图形化的界面，没什么学的，后面用到的时候学习就可以

#Gazebo是用于模拟真实环境生产数据的，边用边学

以上就是要学习的所有ROS2知识，对于URDF之类的其他相关知识，目前认为是没有必要学习，真要是以后有需要再学也不迟